# California State Procurement Data — Exploration

In [26]:
import pandas as pd
from pymongo import MongoClient

df = pd.read_csv(r"C:\Users\SP7\Downloads\archive (1)\PURCHASE ORDER DATA EXTRACT 2012-2015_0.csv", low_memory=False)

## 1. Basic Info

In [27]:
print(df.shape)

(346018, 31)


We have **346,018 purchase orders** and **31 columns** describing each one.

In [28]:
print(df.dtypes)

Creation Date               object
Purchase Date               object
Fiscal Year                 object
LPA Number                  object
Purchase Order Number       object
Requisition Number          object
Acquisition Type            object
Sub-Acquisition Type        object
Acquisition Method          object
Sub-Acquisition Method      object
Department Name             object
Supplier Code              float64
Supplier Name               object
Supplier Qualifications     object
Supplier Zip Code           object
CalCard                     object
Item Name                   object
Item Description            object
Quantity                   float64
Unit Price                  object
Total Price                 object
Classification Codes        object
Normalized UNSPSC          float64
Commodity Title             object
Class                      float64
Class Title                 object
Family                     float64
Family Title                object
Segment             

Most columns are stored as text (`object`). The problem ones are:
- **Total Price** and **Unit Price** — stored as text because of the `$` sign. We need them as numbers to do any math.
- **Creation Date** and **Purchase Date** — stored as text, not real dates. We need to convert them to work with time-based queries.
- **Quantity** is already a number, which is fine.

In [29]:
df.head(5)

,Creation Date,Purchase Date,Fiscal Year,LPA Number,Purchase Order Number,Requisition Number,Acquisition Type,Sub-Acquisition Type,Acquisition Method,Sub-Acquisition Method,...,Classification Codes,Normalized UNSPSC,Commodity Title,Class,Class Title,Family,Family Title,Segment,Segment Title,Location
0,08/27/2013,NaN,2013-2014,7-12-70-26,REQ0011118,REQ0011118,IT Goods,NaN,WSCA/Coop,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/29/2014,NaN,2013-2014,NaN,REQ0011932,REQ0011932,NON-IT Goods,NaN,Informal Competitive,NaN,...,76121504,76121504.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11/01/2013,NaN,2013-2014,NaN,REQ0011476,REQ0011476,IT Services,NaN,Informal Competitive,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"95841\n(38.662263, -121.346136)"
3,06/13/2014,06/05/2014,2013-2014,NaN,4500236642,NaN,NON-IT Goods,NaN,Informal Competitive,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"91436\n(34.151642, -118.49051)"
4,03/12/2014,03/12/2014,2013-2014,1-10-75-60A,4500221028,NaN,NON-IT Goods,NaN,Statewide Contract,NaN,...,44103127,44103127.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"95814\n(38.580427, -121.494396)"


A quick look at the first 5 rows. You can see dates like `08/27/2013` and prices like `$1.00` — both stored as plain text and need to be cleaned.

## 2. Data Quality

In [30]:
df.isnull().sum()

Creation Date                   0
Purchase Date               17436
Fiscal Year                     0
LPA Number                 253673
Purchase Order Number           0
Requisition Number         331649
Acquisition Type                0
Sub-Acquisition Type       277681
Acquisition Method              0
Sub-Acquisition Method     315122
Department Name                 0
Supplier Code                  36
Supplier Name                  36
Supplier Qualifications    204273
Supplier Zip Code           70110
CalCard                         0
Item Name                      32
Item Description              202
Quantity                       30
Unit Price                     30
Total Price                    30
Classification Codes         1017
Normalized UNSPSC            1017
Commodity Title              3295
Class                        3295
Class Title                  3295
Family                       3295
Family Title                 3295
Segment                      3295
Segment Title 

Several columns have missing data worth noting:
- **Purchase Date** — missing for 17,436 rows (~5%). Many orders have no recorded delivery date.
- **Requisition Number** — missing for 331,649 rows (~96%). Mostly not filled in, so it's not very useful.
- **LPA Number** — missing for 253,673 rows (~73%). Same issue.
- **Item Name** — only 32 missing, which is fine.
- **Total Price / Unit Price** — only 30 missing rows, very clean.

In [31]:
df.duplicated().sum()

np.int64(2084)

There are **2,084 duplicate rows** — exact copies of other rows. We'll remove them during cleaning to avoid counting the same order twice.

## 3. Cleaning

In [32]:
# Remove $ and commas, convert to float
df['Total Price'] = df['Total Price'].str.replace(r'[\$,]', '', regex=True).astype(float)
df['Unit Price']  = df['Unit Price'].str.replace(r'[\$,]', '', regex=True).astype(float)

print(df[['Unit Price', 'Total Price']].head(5))

   Unit Price  Total Price
0        1.00         1.00
1        2.00         4.00
2      150.00       675.00
3         NaN          NaN
4     6080.26      6080.26


Prices are now real numbers. We can now calculate totals, averages, and compare values properly.

In [33]:
# Convert dates to datetime
df['Creation Date'] = pd.to_datetime(df['Creation Date'], errors='coerce')
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], errors='coerce')

print(df[['Creation Date', 'Purchase Date']].head(5))

  Creation Date Purchase Date
0    2013-08-27           NaT
1    2014-01-29           NaT
2    2013-11-01           NaT
3    2014-06-13    2014-06-05
4    2014-03-12    2014-03-12


Dates are now proper date objects. This lets us filter by month, group by quarter, and sort by time. The `Purchase Date` column will still show `NaT` (Not a Time) where values were missing — that's expected.

In [34]:
# Normalize Item Name — lowercase and strip whitespace
df['Item Name'] = df['Item Name'].str.lower().str.strip()

# Strip whitespace from all other string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Drop duplicate rows
df = df.drop_duplicates()

print(f"Rows after removing duplicates: {len(df):,}")

Rows after removing duplicates: 343,884


Item names are now all lowercase, so "Toner" and "toner" will be counted as the same item. We also removed 2,084 duplicate rows. The dataset is now ready for analysis.

In [35]:
# Add helper columns
df['Year']    = df['Creation Date'].dt.year
df['Month']   = df['Creation Date'].dt.month
df['Quarter'] = df['Creation Date'].dt.year.astype(str) + '-Q' + df['Creation Date'].dt.quarter.astype(str)

print(df[['Creation Date', 'Year', 'Month', 'Quarter']].head(5))

  Creation Date  Year  Month  Quarter
0    2013-08-27  2013      8  2013-Q3
1    2014-01-29  2014      1  2014-Q1
2    2013-11-01  2013     11  2013-Q4
3    2014-06-13  2014      6  2014-Q2
4    2014-03-12  2014      3  2014-Q1


We added Year, Month, and Quarter columns. These make it easy to group and filter orders by time period without writing complex date logic every time.

## 4. EDA — Exploratory Data Analysis

In [36]:
print(df['Creation Date'].min())
print(df['Creation Date'].max())

2012-07-02 00:00:00
2015-06-30 00:00:00


The dataset covers **3 full fiscal years**, from July 2012 to June 2015. Any query that filters by date should stay within this range.

In [37]:
print("Departments:", df['Department Name'].nunique())
print("Suppliers:  ", df['Supplier Name'].nunique())

Departments: 111
Suppliers:   24732


There are **111 departments** and **24,732 suppliers** in this dataset. The large number of suppliers suggests many one-off purchases alongside regular vendors.

In [38]:
print(f"Total spending: ${df['Total Price'].sum():,.2f}")

Total spending: $151,183,283,732.63


This is the total amount spent across all purchase orders in the dataset over the 3-year period.

In [39]:
df.groupby('Quarter')['Total Price'].sum().sort_values(ascending=False).head(10)

Quarter
2015-Q2    2.880142e+10
2013-Q1    2.576960e+10
2013-Q2    1.901294e+10
2014-Q2    1.561676e+10
2012-Q3    1.256266e+10
2013-Q3    1.255410e+10
2014-Q1    1.011753e+10
2015-Q1    8.288519e+09
2014-Q3    5.926209e+09
2012-Q4    4.531544e+09
Name: Total Price, dtype: float64

This shows which quarters had the highest total spending. The last quarter of each fiscal year (Q2 = Oct–Dec, Q4 = Apr–Jun) tends to spike as departments rush to spend remaining budgets.

In [40]:
df['Creation Date'].dt.to_period('M').value_counts().sort_index()

Creation Date
2012-07     8402
2012-08     8092
2012-09     7901
2012-10     8159
2012-11     7341
2012-12     7301
2013-01     8079
2013-02     7397
2013-03     9388
2013-04     9353
2013-05    12501
2013-06    14469
2013-07     9417
2013-08     9399
2013-09     9497
2013-10     8780
2013-11     7783
2013-12     8617
2014-01     7374
2014-02     8358
2014-03    10322
2014-04    11558
2014-05    11250
2014-06    17536
2014-07     9430
2014-08     9211
2014-09    10180
2014-10     9558
2014-11     7067
2014-12     7624
2015-01     7537
2015-02     7104
2015-03     9863
2015-04    11547
2015-05    11006
2015-06    15483
Freq: M, Name: count, dtype: int64

Monthly order counts show seasonal patterns. June consistently has the highest volume — the end of California's fiscal year — when departments place last-minute orders.

In [41]:
df['Item Name'].value_counts().head(20)

Item Name
medical supplies                        3837
contract                                3103
toner                                   1894
ew                                      1605
office supplies                         1568
expert witness                          1420
medical vocational training services    1126
medical vocational training             1111
dental supplies                         1072
tra                                      918
vocational rehabilitation services       902
software                                 809
water tender                             739
hired equipment                          733
chairs                                   721
service contract                         706
software maintenance                     633
asphalt                                  628
toners                                   621
ca recycle fee                           586
Name: count, dtype: int64

The most ordered items are **medical supplies, toner, and office supplies** — everyday operational items. After lowercasing, "Toner" and "toner" now correctly merge into one group.

In [42]:
df.groupby('Department Name')['Total Price'].sum().sort_values(ascending=False).head(10)

Department Name
Health Care Services, Department of              9.975931e+10
Public Health, Department of                     5.619733e+09
Social Services, Department of                   5.565307e+09
Corrections and Rehabilitation, Department of    4.692279e+09
State Hospitals, Department of                   4.545295e+09
Transportation, Department of                    4.343621e+09
High Speed Rail Authority, California            3.565362e+09
Water Resources, Department of                   2.781368e+09
Correctional Health Care Services                2.640573e+09
Employment Development Department                1.724896e+09
Name: Total Price, dtype: float64

The **Health & Human Services Agency** and **Department of Transportation** are the biggest spenders by far. This makes sense — they manage large contracts and infrastructure projects.

In [43]:
df.groupby('Supplier Name')['Total Price'].sum().sort_values(ascending=False).head(10)

Supplier Name
Health Net Community Solutions, Inc.                  1.358706e+10
L.A. Care Health Plan                                 1.116013e+10
Delta Dental of California                            8.172038e+09
Blue Cross of California Partnership Plan, Inc.       7.176561e+09
Inland Empire Health Plan                             5.438790e+09
County of Los Angeles                                 4.409261e+09
Fresno-Kings-Madera Regional Health Authority         3.509792e+09
Health Plan of San Joaquin                            3.102766e+09
Ventura County                                        3.029475e+09
Molina Healthcare of California Partner Plan, Inc.    3.026911e+09
Name: Total Price, dtype: float64

The top suppliers by total spend. Some names at the top may have inflated totals due to data entry issues (e.g. capped values like `$999,999`). Worth keeping in mind when building spending reports.

## Load to MongoDB

In [44]:
# Convert dates to strings for clean MongoDB storage
df['Creation Date'] = df['Creation Date'].dt.strftime('%Y-%m-%d')
df['Purchase Date'] = df['Purchase Date'].dt.strftime('%Y-%m-%d')

client     = MongoClient('mongodb://localhost:27017/')
collection = client['procurement_db']['orders']

collection.drop()  # clear collection before re-inserting

records = df.to_dict(orient='records')
result  = collection.insert_many(records)

print(f"Inserted {len(result.inserted_ids):,} records into procurement_db.orders")
client.close()

ServerSelectionTimeoutError: localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69e93dd78dcd3e939ee5f856, topology_type: Unknown, servers: [<ServerDescription ('localhost', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:27017: [WinError 10061] No connection could be made because the target machine actively refused it (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

All cleaned records are now stored in MongoDB under `procurement_db.orders`. The collection is ready to be queried by the procurement assistant.